# Generate 15 bp barcodes with Hamming distance >= 3

- Generate 48,000 barcodes
- Barcode length: 15 bp
- Minimum pairwise Hamming distance: 3
- Max GC fraction: 0.70
- Exclude barcodes that create a restriction site **overlapping the barcode**
  in the construct context `RE1 + RE2 | barcode | BG3`, on either strand
- Random seed is fixed for reproducibility

---

## 腳本說明

**目的**：產生一組互相區隔良好的 DNA barcode（條碼），供啟動子文庫的樣本標記／定序後 demultiplex 使用。任兩條條碼至少相差 `MIN_HAMMING` 個鹼基，讓定序或合成錯誤不容易把一條條碼誤讀成另一條。

**產出**：CSV 檔 `outputs/05_barcodes.csv`，共 `N_BARCODES` 列，欄位：
- `candidate_id` — `BC-00001`、`BC-00002` …（06 用這個對應）
- `barcode_id` — 流水號（`BC00001`、`BC00002` …）
- `barcode` — 15 bp 序列（僅 A/C/G/T）
- `gc_fraction` — 該序列的 GC 比例
- `barcode_length` / `min_hamming` / `spec_name` — 規格回寫
- `re_context_prefix` / `re_context_suffix` — 過濾時所用的上下文（見下）

**生成方法**：隨機抽一條序列 → 依序過濾（在 `RE1+RE2 | barcode | BG3` 上下文中排除**壓到 barcode**的限制酶位點、GC 比例超標）→ 用「Hamming 半徑內鄰居查表」判斷是否與**已收條碼**太接近，全部通過才收下；重複到湊滿 `N_BARCODES` 條。整段流程用固定亂數種子，結果可完全重現。

> **上下文過濾**：barcode 不會單獨存在，06 會把它夾在 RE2 和 BG3 中間。只檢查裸 15-mer 的舊版本，讓 1,546 個切位從兩個接縫溜進文庫（其中 125 個是 BG5/BG3 內建的 Eco31I/BsaI，會直接破壞 Golden Gate 組裝）。現在檢查 `CONTEXT_PREFIX + barcode + CONTEXT_SUFFIX`，但**只算跨過 barcode 邊界的位點**——flank 內部本來就有設計好的切位，那些不是 barcode 的問題。

**可調整變數（都在下一格 code）**：
- `N_BARCODES` — 要產生的條碼數（目前 48,000）
- `BARCODE_LEN` — 條碼長度 bp（目前 15）
- `MIN_HAMMING` — 任兩條最小 Hamming distance（目前 3）；`CONFLICT_RADIUS` 會自動 = `MIN_HAMMING - 1`
- `MAX_GC_FRACTION` — GC 比例上限（目前 0.70）
- `RANDOM_SEED` — 亂數種子；換掉會得到不同、但同樣合格的一組條碼
- `MAX_ATTEMPTS` — 嘗試次數上限；湊不滿會直接報錯
- `OUTPUT_PATH` — 輸出檔路徑（由參數推導，不寫死檔名）
- `RE_SITES` — 要排除的限制酶辨識位點表（在下下格）
- `CONTEXT_PREFIX` / `CONTEXT_SUFFIX` — 上下文；**必須和 06 cell 3 的 `RE1+RE2`、`BG3` 開頭一致**（06 會檢查）

> 註：調小 `MIN_HAMMING` 或 `BARCODE_LEN`、調高 `MAX_GC_FRACTION` 會讓合格空間變大、生成更快；反之會變慢，極端時可能湊不滿而觸發 `MAX_ATTEMPTS` 報錯。

## 1. 參數設定與匯入

設定所有可調參數並建立亂數產生器。要改條碼規格（數量、長度、HD、GC 上限、種子、輸出路徑）都集中在這一格。

- `CONFLICT_RADIUS = MIN_HAMMING - 1`：後面判斷「太接近」用的半徑，改 `MIN_HAMMING` 就會自動連動。
- `rng = random.Random(RANDOM_SEED)`：獨立亂數器，固定種子確保結果可重現。
- `BASES = 'ACGT'`：條碼使用的鹼基字元集。


In [1]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


PROJECT_ROOT : C:\project\Whole-model
DATA_DIR     : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data
RELEASE_OUT  : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs


In [2]:
from itertools import combinations
from pathlib import Path
import random

import pandas as pd

N_BARCODES = 48_000
BARCODE_LEN = 15
MIN_HAMMING = 3
CONFLICT_RADIUS = MIN_HAMMING - 1
MAX_GC_FRACTION = 0.70
RANDOM_SEED = 42
MAX_ATTEMPTS = 500_000_000

# Derived from the parameters above, never hardcoded: the previous literal
# filename is how a 12 bp run came to be stored as 'barcodes_15bp_HD3'.
# Writes into the release outputs/ folder; tables/ is left untouched.
SPEC_NAME = f'barcodes_{BARCODE_LEN}bp_HD{MIN_HAMMING}_no_RE_sites'
OUTPUT_PATH = RELEASE_OUT / '05_barcodes.csv'

rng = random.Random(RANDOM_SEED)
BASES = 'ACGT'


## 2. 限制酶位點表與輔助函式

定義後續選殖要避開的限制酶辨識序列，以及過濾用的小工具。

- `RE_SITES`：限制酶名稱 → 辨識序列的對照表。
- `revcomp(seq)`：回傳序列的反向互補股。
- `has_re_site(seq)`：檢查序列**本身與其反向互補股**是否含任一位點（雙股都查）。
- `has_re_site_in_context(barcode)`：**實際使用的過濾器**。先把 barcode 接上
  `CONTEXT_PREFIX`（= 06 的 `RE1 + RE2`）與 `CONTEXT_SUFFIX`（= 06 的 BG3 前 7 nt）再檢查，
  這樣跨接縫產生的位點才抓得到。
  **只有壓到 barcode 的位點才算**：flank 本身就帶著設計好的位點（`CONTEXT_PREFIX`
  結尾就是 XbaI，因為 RE2 就是 XbaI），若直接對整條字串做 substring 比對，
  每一條候選都會被打掉。這和 07 在組好的構築上做的 expected / unexpected 判定同一套邏輯。
  前後各 7 nt 是能抓到最長位點（NotI，8 bp）只壓到 barcode 一個 base 的最小長度。
- `random_barcode(length)`：隨機抽一條長度為 `length` 的序列。
- `gc_fraction(seq)` / `passes_gc_filter(seq)`：計算 GC 比例，並判斷是否 ≤ `MAX_GC_FRACTION`。

In [3]:
# Restriction enzyme recognition sites to avoid.
RE_SITES = {
    'BamHI':   'GGATCC',
    'BcuI':    'ACTAGT',   # = SpeI
    'BglII':   'AGATCT',
    'Eco31I':  'GGTCTC',   # = BsaI
    'EcoRI':   'GAATTC',
    'HindIII': 'AAGCTT',
    'KpnI':    'GGTACC',
    'MluI':    'ACGCGT',
    'NcoI':    'CCATGG',
    'NdeI':    'CATATG',
    'NheI':    'GCTAGC',
    'NotI':    'GCGGCCGC',
    'PstI':    'CTGCAG',
    'SacI':    'GAGCTC',
    'SalI':    'GTCGAC',
    'SmaI':    'CCCGGG',
    'VspI':    'ATTAAT',   # = AseI
    'XbaI':    'TCTAGA',
    'XhoI':    'CTCGAG',
}

_comp = str.maketrans('ACGT', 'TGCA')

def revcomp(seq):
    return seq.translate(_comp)[::-1]

def has_re_site(seq, re_sites=RE_SITES, check_revcomp=True):
    seq = str(seq).upper()
    targets = [seq]
    if check_revcomp:
        targets.append(revcomp(seq))

    for target in targets:
        for site in re_sites.values():
            if site in target:
                return True
    return False

# A barcode is never synthesised on its own: 06 puts it between RE2 and BG3, so a
# site can appear across either junction even when the barcode itself is clean.
# Screening only the bare 15-mer is what let 1,546 junction sites (125 of them
# Eco31I/BsaI, the Golden Gate enzyme built into BG5/BG3) into the previous
# library.
#
# Only sites that OVERLAP the barcode may count. The flanks carry sites by
# design - CONTEXT_PREFIX literally ends in XbaI, because RE2 is XbaI - so a
# plain substring test on the flanked string rejects every candidate. This is
# the same expected/unexpected split 07 applies to the assembled construct.
#
# The two context strings must stay equal to RE1 + RE2 and the head of BG3 in
# 06 cell 3; both are written into 05_barcodes.csv so 06 can verify the pair.
# 7 nt on each side is the minimum that still catches the longest site
# (NotI, 8 bp) overlapping the barcode by a single base.
CONTEXT_PREFIX = 'GTCGACTCTAGA'   # RE1 + RE2 (SalI + XbaI), upstream of the barcode
CONTEXT_SUFFIX = 'GGTCTCA'        # first 7 nt of BG3, downstream

def has_re_site_in_context(barcode, re_sites=RE_SITES):
    """True if `barcode` creates an RE site once flanked the way 06 flanks it."""
    context = CONTEXT_PREFIX + barcode + CONTEXT_SUFFIX
    lo, hi = len(CONTEXT_PREFIX), len(CONTEXT_PREFIX) + len(barcode)
    for site in re_sites.values():
        for pattern in {site, revcomp(site)}:
            start = 0
            while True:
                i = context.find(pattern, start)
                if i < 0:
                    break
                if i < hi and i + len(pattern) > lo:   # the hit touches the barcode
                    return True
                start = i + 1
    return False

def random_barcode(length=BARCODE_LEN):
    return ''.join(rng.choices(BASES, k=length))

def gc_fraction(seq):
    seq = str(seq).upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

def passes_gc_filter(seq, max_gc_fraction=MAX_GC_FRACTION):
    return gc_fraction(seq) <= max_gc_fraction


## 3. Hamming 鄰居枚舉（核心去重邏輯）

`neighbors_within_radius(seq, radius)`：列出所有與 `seq` 的 Hamming distance ≤ `radius` 的序列（含 `seq` 自己）。

主迴圈用它反查——如果候選條碼的任一個鄰居**已經被收過**，代表兩者距離 ≤ `radius`（即 < `MIN_HAMMING`），就淘汰。這樣就能保證留下來的條碼**任兩條距離 ≥ `MIN_HAMMING`**。

作法：對每個距離 `d`（1 到 `radius`），列舉要更動的 `d` 個位置組合，把這些位置換成其他鹼基後逐一 `yield`。半徑越大、序列越長，枚舉數量增長很快（所以 HD 越大跑越慢）。


In [4]:
def neighbors_within_radius(seq, radius=CONFLICT_RADIUS):
    seq = list(seq)
    length = len(seq)

    if radius < 0:
        return

    yield ''.join(seq)

    for distance in range(1, radius + 1):
        for positions in combinations(range(length), distance):
            originals = [seq[pos] for pos in positions]

            def mutate_position(k):
                if k == distance:
                    yield ''.join(seq)
                    return

                pos = positions[k]
                original = originals[k]
                for base in BASES:
                    if base == original:
                        continue
                    seq[pos] = base
                    yield from mutate_position(k + 1)
                seq[pos] = original

            yield from mutate_position(0)


## 4. 主生成迴圈

反覆隨機抽候選條碼，依序通過三道關卡才收下：

1. **不含限制酶位點**（雙股，`has_re_site`）
2. **GC 比例合格**（`passes_gc_filter`，≤ `MAX_GC_FRACTION`）
3. **與已收條碼距離夠遠**（`neighbors_within_radius` 反查，確保 ≥ `MIN_HAMMING`）

每收滿 1,000 條印一次進度（含累計嘗試次數）。收滿 `N_BARCODES` 或超過 `MAX_ATTEMPTS` 就停；若沒收滿會 `raise RuntimeError`。最後整理成 DataFrame，附上 `barcode_id` 與 `gc_fraction`。


In [5]:
accepted = []
accepted_set = set()
attempts = 0

while len(accepted) < N_BARCODES and attempts < MAX_ATTEMPTS:
    attempts += 1
    candidate = random_barcode()

    if has_re_site_in_context(candidate):
        continue

    if not passes_gc_filter(candidate):
        continue

    if any(neighbor in accepted_set for neighbor in neighbors_within_radius(candidate)):
        continue

    accepted.append(candidate)
    accepted_set.add(candidate)

    if len(accepted) % 1000 == 0:
        print(f'{len(accepted)} / {N_BARCODES}, attempts={attempts}')

if len(accepted) < N_BARCODES:
    raise RuntimeError(f'Only generated {len(accepted)} barcodes after {attempts} attempts')

barcodes = pd.DataFrame({
    'barcode_id': [f'BC{i + 1:05d}' for i in range(len(accepted))],
    'barcode': accepted,
})
barcodes['gc_fraction'] = [gc_fraction(seq) for seq in barcodes['barcode']]

print(f'Generated {len(barcodes)} barcodes after {attempts} attempts')
barcodes.head()


1000 / 48000, attempts=1167


2000 / 48000, attempts=2346


3000 / 48000, attempts=3513


4000 / 48000, attempts=4698


5000 / 48000, attempts=5861


...

45000 / 48000, attempts=53751


46000 / 48000, attempts=54967


47000 / 48000, attempts=56181


48000 / 48000, attempts=57394
Generated 48000 barcodes after 57394 attempts


,barcode_id,barcode,gc_fraction
0,BC00001,GACAGGTACAAGAAG,0.466667
1,BC00002,GAGTATGCATCAATG,0.400000
2,BC00003,TGGTCGTGTGGAACA,0.533333
3,BC00004,AACGCCACTGGAGAC,0.600000
4,BC00005,TGGGTTAACCATTCG,0.466667


## 5. 驗證

對產出結果做完整自我檢查，任一項不過就 `raise`：

- 數量正確、序列**唯一無重複**
- 長度都是 `BARCODE_LEN`、字元只含 A/C/G/T
- 全部**無限制酶位點**、**GC 合格**
- 用鄰居查表做**完整的 Hamming distance 驗證**，確認任兩條距離都 ≥ `MIN_HAMMING`

（這格是獨立於生成迴圈的把關，避免邏輯有漏。）


In [6]:
barcode_set = set(barcodes['barcode'])

assert len(barcodes) == N_BARCODES
assert len(barcode_set) == N_BARCODES
assert all(len(seq) == BARCODE_LEN for seq in barcodes['barcode'])
assert all(set(seq) <= set(BASES) for seq in barcodes['barcode'])
assert all(not has_re_site(seq) for seq in barcodes['barcode'])
assert all(not has_re_site_in_context(seq) for seq in barcodes['barcode'])
assert all(passes_gc_filter(seq) for seq in barcodes['barcode'])

# Full Hamming distance validation using radius lookup.
for seq in barcodes['barcode']:
    barcode_set.remove(seq)
    has_conflict = any(neighbor in barcode_set for neighbor in neighbors_within_radius(seq))
    barcode_set.add(seq)
    if has_conflict:
        raise ValueError(f'Hamming distance conflict found around {seq}')

print(f'Validation passed: unique, no RE sites in {CONTEXT_PREFIX}|barcode|{CONTEXT_SUFFIX} context, full HD >= {MIN_HAMMING}')


Validation passed: unique, no RE sites in GTCGACTCTAGA|barcode|GGTCTCA context, full HD >= 3


## 6. 輸出存檔

寫出標準化的條碼表到 `OUTPUT_PATH`（`outputs/05_barcodes.csv`），並在寫之前再做一次硬性把關：
數量、長度、字母表、重複、以及上下文中的限制酶位點，任一項不符就 `raise`，不會產出標示錯誤的檔案。

⚠️ 重跑這本 notebook 會**整批換掉** barcode（只要 `RANDOM_SEED` 或任何過濾條件改動）。
06 是依 `candidate_id` 順序配對的，所以 barcode↔candidate 的對應也會全部改變。
若已經照舊 barcode 下單或標記過樣本，不要重跑。

In [7]:
# === Standardised barcode table (05_barcodes.csv) ===
# Hard gate: what is written must match what the parameters claim. A length or
# alphabet mismatch raises instead of producing a silently mislabelled file.
observed_lengths = sorted(barcodes['barcode'].str.len().unique())
if observed_lengths != [BARCODE_LEN]:
    raise ValueError(
        f'barcode length mismatch: parameters say {BARCODE_LEN} bp '
        f'({SPEC_NAME}), generated {observed_lengths}'
    )
if len(barcodes) != N_BARCODES:
    raise ValueError(f'expected {N_BARCODES} barcodes, got {len(barcodes)}')
if any(has_re_site_in_context(b) for b in barcodes['barcode']):
    raise ValueError(
        f'a barcode creates an RE site in the {CONTEXT_PREFIX}|barcode|{CONTEXT_SUFFIX} context'
    )
if not barcodes['barcode'].str.fullmatch('[ACGT]+').all():
    raise ValueError('non-ACGT characters in generated barcodes')
if barcodes['barcode'].duplicated().any():
    raise ValueError('duplicate barcodes generated')

barcodes = barcodes.copy()
barcodes.insert(0, 'candidate_id', [f'BC-{i:05d}' for i in range(1, len(barcodes) + 1)])
barcodes['barcode_length'] = BARCODE_LEN
barcodes['min_hamming'] = MIN_HAMMING
barcodes['spec_name'] = SPEC_NAME
barcodes['re_context_prefix'] = CONTEXT_PREFIX
barcodes['re_context_suffix'] = CONTEXT_SUFFIX

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
barcodes.to_csv(OUTPUT_PATH, index=False)
print(f'saved -> {OUTPUT_PATH}')
print(f'spec: {SPEC_NAME}  |  {len(barcodes)} barcodes x {BARCODE_LEN} bp, HD >= {MIN_HAMMING}')
print(barcodes.head(3).to_string(index=False))


saved -> C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\05_barcodes.csv
spec: barcodes_15bp_HD3_no_RE_sites  |  48000 barcodes x 15 bp, HD >= 3
candidate_id barcode_id         barcode  gc_fraction  barcode_length  min_hamming                     spec_name re_context_prefix re_context_suffix
    BC-00001    BC00001 GACAGGTACAAGAAG     0.466667              15            3 barcodes_15bp_HD3_no_RE_sites      GTCGACTCTAGA           GGTCTCA
    BC-00002    BC00002 GAGTATGCATCAATG     0.400000              15            3 barcodes_15bp_HD3_no_RE_sites      GTCGACTCTAGA           GGTCTCA
    BC-00003    BC00003 TGGTCGTGTGGAACA     0.533333              15            3 barcodes_15bp_HD3_no_RE_sites      GTCGACTCTAGA           GGTCTCA
